In [1]:
import pandas as pd
import numpy as np


In [3]:
df = pd.read_csv("data_perparation/prepared_dataset/Hourly_final_cleaned.csv")

In [5]:
df["datetime"] = pd.to_datetime(df["datetime"])
df["date"] = df["datetime"].dt.date

In [6]:
daily = df.groupby(["Id", "date"]).agg({
    "StepTotal": "sum",
    "total_sleep_min": "sum",
    "hourly_heart_rate": "mean",
    "Calories": "sum"
}).reset_index()

In [7]:
daily = daily.rename(columns={
    "StepTotal": "daily_steps",
    "total_sleep_min": "daily_sleep_min",
    "hourly_heart_rate": "avg_heart_rate",
    "Calories": "daily_calories"
})

In [8]:
daily["lack_of_exercise"] = (daily["daily_steps"] < 5000).astype(int)
daily["poor_sleep"] = (daily["daily_sleep_min"] < 360).astype(int)
daily["high_heart_rate"] = (daily["avg_heart_rate"] > 85).astype(int)
daily["high_calorie"] = (daily["daily_calories"] > 3000).astype(int)

daily.head()

,Id,date,daily_steps,daily_sleep_min,avg_heart_rate,daily_calories,lack_of_exercise,poor_sleep,high_heart_rate,high_calorie
0,2022484408,2016-04-01,13603.0,0.0,90.775787,1898.0,0,1,1,0
1,2022484408,2016-04-02,5477.0,0.0,71.553343,1294.0,0,1,0,0
2,2022484408,2016-04-03,11144.0,0.0,75.162948,1711.0,0,1,0,0
3,2022484408,2016-04-04,15313.0,0.0,77.346908,1834.0,0,1,0,0
4,2022484408,2016-04-05,10805.0,0.0,83.283095,2114.0,0,1,0,0


In [9]:
label_cols = [
    "lack_of_exercise",
    "poor_sleep",
    "high_heart_rate",
    "high_calorie"
]

daily[label_cols].sum()

lack_of_exercise    105
poor_sleep          262
high_heart_rate      91
high_calorie         77
dtype: int64

In [10]:
df_labeled = df.merge(
    daily[["Id", "date"] + label_cols],
    on=["Id", "date"],
    how="left"
)

df_labeled.head()

,Id,datetime,StepTotal,total_sleep_min,hourly_heart_rate,Calories,date,lack_of_exercise,poor_sleep,high_heart_rate,high_calorie
0,2022484408,2016-04-01 07:00:00,123.0,0.0,100.326531,78.0,2016-04-01,0,1,1,0
1,2022484408,2016-04-01 08:00:00,440.0,0.0,71.396419,115.0,2016-04-01,0,1,1,0
2,2022484408,2016-04-01 09:00:00,4994.0,0.0,121.205323,427.0,2016-04-01,0,1,1,0
3,2022484408,2016-04-01 10:00:00,2926.0,0.0,97.129771,248.0,2016-04-01,0,1,1,0
4,2022484408,2016-04-01 11:00:00,0.0,0.0,90.000000,62.0,2016-04-01,0,1,1,0


In [11]:
df_labeled.shape

(8499, 11)

In [12]:
df_labeled.to_csv('daily_health.csv', index=False)